# Data Science & Database Assignment: Session 4
## Topic: SELECT Statement Basics

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Core Concepts:** `SELECT`, Column Projection, `LIMIT`, `DISTINCT`, Column Aliasing (`AS`), Execution Order & Debugging  
**Status:** Completed & Pre-Executed  

---

### SQL DQL Execution Lifecycle

While SQL queries are written starting with `SELECT`, relational database engines evaluate clauses in a specific logical order:

```text
1. FROM / JOIN     --> Identify source tables and joins
2. WHERE           --> Filter rows based on conditions
3. GROUP BY        --> Group records
4. HAVING          --> Filter grouped rows
5. SELECT          --> Project specific columns and evaluate expressions
6. DISTINCT        --> Eliminate duplicate result rows
7. ORDER BY        --> Sort final rows
8. LIMIT / OFFSET  --> Restrict number of returned rows
```


## Task 1: Create `MusicPlaylist` Table & Retrieve All Songs
> **Problem Statement:** Create a table named `MusicPlaylist` with columns: `id`, `song_name`, `artist`, `genre`, and `duration`. Insert at least 5 records representing songs from your favorite Spotify playlist, then write a `SELECT` statement to retrieve all columns for all songs.

### Schema Definition
```sql
CREATE TABLE MusicPlaylist (
    id INT PRIMARY KEY,
    song_name VARCHAR(255) NOT NULL,
    artist VARCHAR(255) NOT NULL,
    genre VARCHAR(100) NOT NULL,
    duration VARCHAR(20) NOT NULL
);
```

### Retrieval Query
```sql
SELECT * FROM MusicPlaylist;
```


In [1]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect("music_and_food.db")
cursor = conn.cursor()

# Create table
cursor.execute("DROP TABLE IF EXISTS MusicPlaylist;")
cursor.execute("""
CREATE TABLE MusicPlaylist (
    id INTEGER PRIMARY KEY,
    song_name VARCHAR(255) NOT NULL,
    artist VARCHAR(255) NOT NULL,
    genre VARCHAR(100) NOT NULL,
    duration VARCHAR(20) NOT NULL
);
""")

# Insert 5 Spotify favorite songs
spotify_songs = [
    (1, 'Blinding Lights', 'The Weeknd', 'Synthwave / Pop', '3:20'),
    (2, 'Starboy', 'The Weeknd', 'R&B / Pop', '3:50'),
    (3, 'Shape of You', 'Ed Sheeran', 'Pop', '3:53'),
    (4, 'Levitating', 'Dua Lipa', 'Disco-Pop', '3:23'),
    (5, 'Tum Hi Ho', 'Arijit Singh', 'Bollywood Romantic', '4:22')
]
cursor.executemany("INSERT INTO MusicPlaylist VALUES (?, ?, ?, ?, ?);", spotify_songs)
conn.commit()

# Execute SELECT *
df_all_songs = pd.read_sql_query("SELECT * FROM MusicPlaylist;", conn)
print("Task 1 Output: All Columns for All Songs (SELECT *)")
display(df_all_songs)


Task 1 Output: All Columns for All Songs (SELECT *)


,id,song_name,artist,genre,duration
0,1,Blinding Lights,The Weeknd,Synthwave / Pop,3:20
1,2,Starboy,The Weeknd,R&B / Pop,3:50
2,3,Shape of You,Ed Sheeran,Pop,3:53
3,4,Levitating,Dua Lipa,Disco-Pop,3:23
4,5,Tum Hi Ho,Arijit Singh,Bollywood Romantic,4:22


## Task 2: Column Projection & Row Restriction (`LIMIT 3`)
> **Problem Statement:** Write a SQL query to display only the `song_name` and `artist` columns from the `MusicPlaylist` table, showing just the first 3 records using the `LIMIT` keyword.

### SQL Query:
```sql
SELECT song_name, artist
FROM MusicPlaylist
LIMIT 3;
```
* **Column Projection:** By specifying `song_name, artist` instead of `*`, we reduce I/O overhead and retrieve only the required fields.
* **Row Restriction (`LIMIT 3`):** Tells the database query planner to stop scanning and return as soon as 3 matching records are found.


In [2]:
query_limit3 = """
SELECT song_name, artist
FROM MusicPlaylist
LIMIT 3;
"""

df_limit3 = pd.read_sql_query(query_limit3, conn)
print("Task 2 Output: First 3 songs with song_name and artist only")
display(df_limit3)


Task 2 Output: First 3 songs with song_name and artist only


,song_name,artist
0,Blinding Lights,The Weeknd
1,Starboy,The Weeknd
2,Shape of You,Ed Sheeran


## Task 3: `FoodOrders` Table & Unique Restaurants (`DISTINCT`)
> **Problem Statement:** Suppose you have a table named `FoodOrders` with columns: `id`, `restaurant`, `food_item`, and `order_date`. Write a SQL query to list all unique restaurant names where you have placed orders, using the `DISTINCT` keyword.

### Table Schema & Query:
```sql
CREATE TABLE FoodOrders (
    id INT PRIMARY KEY,
    restaurant VARCHAR(255) NOT NULL,
    food_item VARCHAR(255) NOT NULL,
    order_date DATE NOT NULL
);

SELECT DISTINCT restaurant
FROM FoodOrders;
```
* **`DISTINCT` Keyword:** Evaluates the selected column values and suppresses duplicates, returning only singular unique occurrences.


In [3]:
# Create FoodOrders table
cursor.execute("DROP TABLE IF EXISTS FoodOrders;")
cursor.execute("""
CREATE TABLE FoodOrders (
    id INTEGER PRIMARY KEY,
    restaurant VARCHAR(255) NOT NULL,
    food_item VARCHAR(255) NOT NULL,
    order_date DATE NOT NULL
);
""")

# Insert orders with duplicate restaurant entries to demonstrate DISTINCT
sample_orders = [
    (1, 'Biryani Blues', 'Hyderabadi Dum Biryani', '2026-10-20'),
    (2, "Domino's Pizza", 'Farmhouse Pizza', '2026-10-21'),
    (3, 'Biryani Blues', 'Chicken Tikka', '2026-10-22'),
    (4, 'Punjab Grill', 'Dal Makhani', '2026-10-23'),
    (5, "Domino's Pizza", 'Garlic Bread', '2026-10-24'),
    (6, "Haldiram's", 'Chole Bhature', '2026-10-25')
]
cursor.executemany("INSERT INTO FoodOrders VALUES (?, ?, ?, ?);", sample_orders)
conn.commit()

# Query distinct restaurants
df_distinct_restaurants = pd.read_sql_query("SELECT DISTINCT restaurant FROM FoodOrders;", conn)
print(f"Total orders in table: {len(sample_orders)}")
print(f"Unique restaurants found: {len(df_distinct_restaurants)}")
display(df_distinct_restaurants)


Total orders in table: 6
Unique restaurants found: 4


,restaurant
0,Biryani Blues
1,Domino's Pizza
2,Punjab Grill
3,Haldiram's


## Task 4: Column Aliasing (`AS 'Dish'` and `AS 'Date Ordered'`)
> **Problem Statement:** Write a SQL query on the `FoodOrders` table to select `food_item` as `'Dish'` and `order_date` as `'Date Ordered'`, displaying only these two columns with the column aliases in the output.

### SQL Query:
```sql
SELECT 
    food_item AS "Dish",
    order_date AS "Date Ordered"
FROM FoodOrders;
```
* **Column Aliases (`AS`):** Renames the output headers of the returned result set for presentation and downstream reporting without altering the underlying table schema.
* Quotes (`""` or `''`) are used when alias names contain spaces (e.g., `"Date Ordered"`).


In [4]:
query_aliases = """
SELECT 
    food_item AS "Dish",
    order_date AS "Date Ordered"
FROM FoodOrders;
"""

df_aliases = pd.read_sql_query(query_aliases, conn)
print("Task 4 Output: Aliased Columns ('Dish' and 'Date Ordered')")
display(df_aliases)


Task 4 Output: Aliased Columns ('Dish' and 'Date Ordered')


,Dish,Date Ordered
0,Hyderabadi Dum Biryani,2026-10-20
1,Farmhouse Pizza,2026-10-21
2,Chicken Tikka,2026-10-22
3,Dal Makhani,2026-10-23
4,Garlic Bread,2026-10-24
5,Chole Bhature,2026-10-25


## Task 5: Debugging & Fixing SQL Query Mistake
> **Problem Statement:** You tried running this query: `SELECT DISTINCT food_item, restaurant FROM FoodOrders LIMIT 2`, but it returns an error or doesn't work as expected. Identify and fix the mistake in the query.  
> *Hint: Check the correct placement and usage of the LIMIT keyword in SQL syntax.*

---

### In-Depth Bug Analysis & Diagnosis

#### 1. Non-Deterministic Output (Missing `ORDER BY`)
* In SQL, relational tables are **unordered sets**. Without an explicit `ORDER BY` clause, the database query optimizer returns records in an arbitrary physical storage sequence.
* A `LIMIT 2` query without `ORDER BY` is non-deterministic: consecutive runs or runs on different hardware/indexing can yield different pairs of rows.
* **The Rule:** In professional SQL, **always pair `LIMIT` with `ORDER BY`**.

#### 2. Multi-RDBMS Syntax Incompatibility (T-SQL vs. MySQL/Postgres)
* If this query is executed on **Microsoft SQL Server (SSMS / T-SQL)**, the `LIMIT` keyword does not exist and throws a direct syntax error:  
  `Msg 156, Level 15, State 1: Incorrect syntax near the keyword 'LIMIT'.`  
  *In T-SQL, row limits are placed immediately after `SELECT` using `TOP`:*
  ```sql
  SELECT DISTINCT TOP 2 food_item, restaurant FROM FoodOrders;
  ```
* If executed on **Oracle Database (prior to 12c)**, `LIMIT` is unrecognized; Oracle uses `ROWNUM <= 2` or `FETCH FIRST 2 ROWS ONLY`.

#### 3. Scope of `DISTINCT`
* Notice that `DISTINCT` applies to the **combination** of `(food_item, restaurant)` tuple, not to `food_item` alone.

---

### Corrected Production Queries

#### Option A: Standard SQL (MySQL / PostgreSQL / SQLite) with Deterministic Ordering
```sql
SELECT DISTINCT food_item, restaurant
FROM FoodOrders
ORDER BY food_item ASC
LIMIT 2;
```

#### Option B: Microsoft SQL Server (T-SQL)
```sql
SELECT DISTINCT TOP 2 food_item, restaurant
FROM FoodOrders
ORDER BY food_item ASC;
```

#### Option C: Oracle SQL (12c+) / ANSI Standard
```sql
SELECT DISTINCT food_item, restaurant
FROM FoodOrders
ORDER BY food_item ASC
FETCH FIRST 2 ROWS ONLY;
```


In [5]:
# Execute corrected deterministic query with ORDER BY and LIMIT 2
corrected_query = """
SELECT DISTINCT food_item, restaurant
FROM FoodOrders
ORDER BY food_item ASC
LIMIT 2;
"""

df_corrected = pd.read_sql_query(corrected_query, conn)
print("Task 5 Output: Corrected Query Result (Ordered & Limited)")
display(df_corrected)

# Cleanly close connection
conn.close()


Task 5 Output: Corrected Query Result (Ordered & Limited)


,food_item,restaurant
0,Chicken Tikka,Biryani Blues
1,Chole Bhature,Haldiram's


---

## 📌 Summary of Core Concepts Mastered

| Concept | Keyword | Purpose | Example |
| :--- | :--- | :--- | :--- |
| **All Columns** | `*` | Project every attribute in table | `SELECT * FROM MusicPlaylist;` |
| **Projection** | `col1, col2` | Retrieve only needed columns | `SELECT song_name, artist FROM MusicPlaylist;` |
| **Row Restriction** | `LIMIT n` | Restrict output row count | `LIMIT 3;` |
| **Deduplication** | `DISTINCT` | Eliminate duplicate rows | `SELECT DISTINCT restaurant FROM FoodOrders;` |
| **Aliasing** | `AS` | Rename output header | `food_item AS "Dish"` |
| **Deterministic Limit** | `ORDER BY ... LIMIT` | Ensure reproducible slice | `ORDER BY food_item ASC LIMIT 2;` |
